In [ ]:
import tqdm
import ffmpeg
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import os


rcParams.update({
    "font.size":        16,        
    "axes.titlesize":   16,
    "axes.labelsize":   16,
    "xtick.labelsize":  16,
    "ytick.labelsize":  16,
    "legend.fontsize":  16,
    "figure.titlesize": 16,
})


os.chdir(os.path.dirname(os.getcwd()))
print(os.getcwd())

In [ ]:
EXPERIMENT_RESULTS_PATH = 'Paper-IS26-experiments/dataset-spanishad/'

In [ ]:
## Change this to process the audio files in your dataset in case that you don't want to use the preprocessed metadata provided in the experiments.
def generate_metadata(base_audio_path):
    audio_files = list(Path(base_audio_path).rglob('*.wav')) + list(Path(base_audio_path).rglob('*.mp3'))
    metadata = []
    for audio_file in tqdm.tqdm(audio_files):
        if ('CTR' in audio_file.name or 'AD' in audio_file.name) and 'lamina_1' in audio_file.stem:
            metadata.append({
                'file': str(audio_file),
                'sample_id': audio_file.stem,
                'condition': 1 if audio_file.name.split('_')[0] == 'AD' else 0,
                'subject': '_'.join(audio_file.stem.split('_')[0:2]),
                'dataset': 'spanishad',
                'subset': 'real_original',
                'experiment_description': 'spanishad\n(real_original)'
            })
    return pd.DataFrame(metadata)

In [ ]:
metadata_list = []
for metadata in Path(EXPERIMENT_RESULTS_PATH).glob('**/metadata.pkl'):
    df = pd.read_pickle(metadata)
    df['dataset'] = metadata.parents[1].name.replace('dataset-', '')
    df['subset'] = metadata.parents[0].name.replace('subset-', '')
    df['experiment_description'] = df['dataset'] + '\n(' + df['subset'] + ')'
    df = df.drop_duplicates('sample_id')
    metadata_list.append(df)

custom_metadata = generate_metadata('audios/')
all_experiments_metadata = pd.concat(metadata_list + [custom_metadata], ignore_index=True)

print(f'Amount of experiments: {len(all_experiments_metadata.experiment_description.unique())}')

In [ ]:
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

summarize_experiments = all_experiments_metadata.groupby(['experiment_description', 'condition']).size().reset_index(name='count')
ax = sns.barplot(data=summarize_experiments, y='experiment_description', x='count', hue='condition', palette='viridis')

plt.title('Sample Distribution by Condition across Datasets and Subsets', fontsize=16, pad=20)
plt.xlabel('Sample Count (Number of sample_id)', fontsize=12)
plt.ylabel('Dataset', fontsize=12)
plt.legend(title='Condition', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
def flatten_data(data):
    reformat_data = {}
    for k, v in data.items():
        if isinstance(v, dict):
            v = {f'{k}_{kv}': vv for kv, vv in v.items()}
            reformat_data.update(flatten_data(v))
        elif isinstance(v, list):
            if len(v) != 1:
                v = {f'{k}_{i}_{kv}': vv for i, v_elem in enumerate(v) for kv, vv in v_elem.items()}
            else:
                v = {f'{k}_{kv}': vv for kv, vv in v[0].items()}
            reformat_data.update(flatten_data(v))
        else:
            reformat_data[k] = v
    return reformat_data

In [ ]:
files_to_analyze = all_experiments_metadata.file.unique()
print(f'Amount of unique audio files: {len(files_to_analyze)}')

audio_metadata = []
for audio_file in tqdm.tqdm(files_to_analyze):
    if Path(audio_file).exists():
        df = flatten_data(ffmpeg.probe(audio_file))
        audio_metadata.append(df)
    else:
        print(f'File not found: {audio_file}')

audio_metadata = pd.DataFrame(audio_metadata)
audio_metadata_columns = list(set(audio_metadata.columns) - set(['format_filename']))
all_experiments_metadata = all_experiments_metadata.merge(audio_metadata, left_on='file', right_on='format_filename', how='left')

In [ ]:
columns = ['dataset', 'subset', 'condition'] + ['streams_codec_name', 'streams_sample_rate', 'streams_channels', 'streams_bits_per_sample']
all_experiments_metadata.groupby(columns).size().reset_index(name='counts')

In [ ]:
all_experiments_results = []
repetitions = 100
for (dataset, subset), exp_audio_metadata in all_experiments_metadata.groupby(['dataset', 'subset']):
    print(f'Running experiment over samples in {dataset} dataset ({subset})')
    y = exp_audio_metadata['condition'].to_numpy()

    for col in audio_metadata_columns:
        if exp_audio_metadata[col].dtype != int:
            le = LabelEncoder()
            exp_audio_metadata[col] = le.fit_transform(exp_audio_metadata[col])

    X = exp_audio_metadata[audio_metadata_columns].to_numpy()

    results = []
    for i in tqdm.tqdm(range(repetitions)):
        train_features, test_features, train_labels, test_labels = train_test_split(X, y, test_size=0.2, random_state=i, stratify=y, shuffle=True)
        clf = RandomForestClassifier(n_estimators=10, random_state=i)
        clf = clf.fit(train_features, train_labels)
        test_predictions = clf.predict_proba(test_features)[:, 1]

        results.append({
            'dataset': dataset,
            'subset': subset,
            'repetition': i,
            'Accuracy': accuracy_score(test_labels, test_predictions > 0.5),
            'AUC': roc_auc_score(test_labels, test_predictions)
        })

    all_experiments_results.extend(results)
all_experiments_results = pd.DataFrame(all_experiments_results)


In [ ]:
for dataset, dataset_df in all_experiments_results.groupby('dataset'):
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(20, 10), squeeze=False, sharey=True, sharex=True)
    for ax, metric in zip(axes.flatten(), ['Accuracy', 'AUC']):
        sns.boxplot(data=dataset_df, x="subset", y=metric, ax=ax, legend=False)
        sns.stripplot(data=dataset_df, x="subset", y=metric, color=".2", alpha=0.5, jitter=True, ax=ax)
        ax.set_ylabel(metric)
        ax.set_xlabel("Subset")
        if metric == 'AUC':
            ax.axhline(0.5, color='indianred', linestyle='--', label='Random Chance')
    fig.suptitle(f"Experiment Results for dataset {dataset.upper()}", fontsize=20, y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.show()
